# Salida final — global, constante y compartida


In [ ]:
%pip install Pillow nvcc4jupyter
%load_ext nvcc4jupyter
!nvidia-smi
!nvcc --version


## Proyecto


In [ ]:
from pathlib import Path
import subprocess, os, json
root = Path('/content/P3-CUDA-Paralela')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git', str(root)], check=True)
os.chdir(root)
subprocess.run(['git', 'checkout', 'main'], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['make', 'build/hough_final'], check=True)
out = Path('results/salida-final')
out.mkdir(parents=True, exist_ok=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True))


## Validación y acumulador final


In [ ]:
import tempfile
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
log += 'Commit: ' + subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True)
log += 'Compilacion: make build/hough_final; -O2 -std=c++17 --fmad=false\n'
log += 'Validacion de salida; las mediciones comparativas estan en fase6-tres-memorias/final.\n'
def validate(image, label, output=None):
    global log
    args = ['./build/hough_final', str(image)]
    if output is not None:
        args += ['--output', str(output)]
    run = subprocess.run(args, text=True, capture_output=True)
    text = label + '\n' + run.stdout + run.stderr
    log += text
    (out / 'salida-final-validacion.txt').write_text(log)
    print(text)
    assert run.returncode == 0, label
validate('data/runway.pgm', 'Imagen original', out)
accumulator = json.loads((out / 'tres-memorias-accumulator.json').read_text())
assert accumulator['variant'] == 'tres_memorias' and accumulator['gpu_executed']
reference = json.loads(Path('results/fase2-global/global-accumulator.json').read_text())
assert accumulator['votes'] == reference['votes']
with tempfile.TemporaryDirectory() as directory:
    for label, w, h, value in [('parcial-21', 7, 3, 255), ('negra', 7, 3, 0),
                               ('parcial-300', 20, 15, 255)]:
        image = Path(directory) / (label + '.pgm')
        image.write_bytes(f'P5\n{w} {h}\n255\n'.encode() + bytes([value]) * (w * h))
        validate(image, label)
log += 'VALIDACION DE SALIDA OK: original, negra, bloques parciales y acumulador global de referencia.\n'
(out / 'salida-final-validacion.txt').write_text(log)
print('VALIDACION DE SALIDA OK')


## Imagen con líneas a color


In [ ]:
from scripts.render_lines import render
from IPython.display import display, Image
THRESHOLD = None
MAX_LINES = 12
details = render('data/runway.pgm', out / 'tres-memorias-accumulator.json',
                 out / 'tres-memorias-lines.png', threshold=THRESHOLD, max_lines=MAX_LINES)
assert details['variant'] == 'tres_memorias' and details['gpu_executed']
display(Image(filename=str(out / 'tres-memorias-lines.png')))


## Evidencia


In [ ]:
from google.colab import files
import zipfile
bundle = Path('build/salida-final.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.glob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(bundle))
